In [1]:
from ProcessBasic import findfiles
from datetime import datetime
import pandas as pd
import os 

def roc_to_ad(date_str: str) -> str:
    """將民國年日期 (格式:YYY.MM.DD) 轉為西元 YYYY-MM-DD 字串"""
    year, month, day = map(int, date_str.split('.'))
    return datetime(year + 1911, month, day).strftime("%Y-%m-%d")

def getinfodata(file):
    sheetname = "路段方位"
    surveypoint = read_specific_data(file, sheetname, 'B1')
    surveyroadname = read_specific_data(file, sheetname, 'B2')
    surveydate = read_specific_data(file, sheetname, 'B6')
    surveydate = roc_to_ad(surveydate)
    return surveypoint, surveyroadname, surveydate

def combine_date_time(date_str, time_val):
    time_val = int(float(time_val))
    hour = time_val // 100
    minute = time_val % 100
    return pd.to_datetime(f"{date_str} {hour:02d}:{minute:02d}")

def getsurveydata(file, surveydate, surveyroadname, surveypoint, startrow=2):
        
    sheetname = "資料輸入"

    # 讀取資料
    df = pd.read_excel(file, sheet_name=sheetname, header = [1,2])
    df = df.iloc[startrow:startrow+64,0:-2]

    # 轉換欄位資料
    df.columns = df.columns.map(lambda x: '_'.join([str(i) for i in x]).replace('\n', ''))
    df.columns.values[0] = '開始時間'
    df.columns.values[1] = '結束時間'
    df = df.drop(columns=[col for col in df.columns if "." in str(col)])

    df['開始時間'] = df['開始時間'].astype('int64')
    df['結束時間'] = df['結束時間'].astype('int64')

    # 把方向轉為欄位資料

    ## 使用 wide_to_long 將資料轉換
    df_long = pd.wide_to_long(
        df,
        stubnames=["往北", "往南"],  # 兩個方向的欄位前綴
        i=["開始時間", "結束時間"],  # 保持的索引
        j="車種",                    # 對應的車種
        sep="_",
        suffix=".+"
    ).reset_index()

    ## 調整欄位
    df_long = pd.melt(
        df_long,
        id_vars=["開始時間", "結束時間", "車種"],
        value_vars=["往北", "往南"],
        var_name="方向",
        value_name="數量"
    )

    ## 重新整理為所需格式
    dfresult = df_long.pivot_table(
        index=["開始時間", "結束時間", "方向"],
        columns="車種",
        values="數量",
        fill_value=0
    ).reset_index()

    dfresult['調查點位'] = surveypoint
    dfresult['路段名稱'] = surveyroadname
    dfresult['調查日期'] = surveydate
    if startrow == 2 :
        dfresult['快慢車道'] = '快車道'
    else:
        dfresult['快慢車道'] = '慢車道'


    # dfresult['開始時間'] = dfresult.apply(lambda r: combine_date_time(r['調查日期'], r['開始時間']), axis=1)
    # dfresult['結束時間'] = dfresult.apply(lambda r: combine_date_time(r['調查日期'], r['結束時間']), axis=1)

    dfresult = dfresult.reindex(columns = ['調查點位', '路段名稱', '調查日期', '開始時間', '結束時間', '快慢車道','方向', '聯結車', '大貨車', '大客車(紅牌)', '大客車(綠牌)',  '小型車', '機車', '自行車', '行人'])

    return dfresult

def hourlyformat(df):
    df = df.copy()
    df['時段'] = df['開始時間'] // 100 
    dfhourly  = df.groupby(['調查點位', '路段名稱', '調查日期', '快慢車道','方向', '時段']).sum().reset_index()
    dfhourly.drop(columns = ['開始時間', '結束時間'], inplace=True)

    return dfhourly

def read_specific_data(excelfilepath, sheetname, cell):
    df = pd.read_excel(excelfilepath, sheet_name=sheetname, header=None)
    
    # cell 轉換
    col = ord(cell[0].upper()) - ord('A')
    row = int(cell[1:]) - 1
    return df.iloc[row, col]

def main():

    project6674folder = r"D:\OneDrive - 鼎漢國際工程顧問股份有限公司\台北鼎漢(B_6600)\6674_海線雙軌可行性"
    surveydfolder = os.path.join(project6674folder, "Technical", "02 交通量調查", "調查結果", "路段交通量")

    dfs = [] 
    for file in findfiles(surveydfolder, "") :
        surveypoint, surveyroadname, surveydate = getinfodata(file)
        dfquick = getsurveydata(file, surveydate, surveyroadname, surveypoint, startrow=2)
        dfslow = getsurveydata(file, surveydate, surveyroadname, surveypoint, startrow=74)
        df = pd.concat([dfquick, dfslow], ignore_index=True)
        dfs.append(df)
    df_original = pd.concat(dfs, ignore_index=True)
    df_hourly = hourlyformat(df_original)

    outputpath = os.path.join(os.getcwd(),"..", "03_前期資料比對", "6674原始交通量彙整.xlsx")
    with pd.ExcelWriter(outputpath) as writer:
        df_original.to_excel(writer, sheet_name="原始交通量資料", index=False)
        df_hourly.to_excel(writer, sheet_name="分時交通量資料", index=False)

if __name__ == '__main__':
    main()

C:\Users\kjchang\AppData\Local\Temp\ipykernel_24936\3431243700.py:64: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfresult = df_long.pivot_table(
C:\Users\kjchang\AppData\Local\Temp\ipykernel_24936\3431243700.py:64: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfresult = df_long.pivot_table(
C:\Users\kjchang\AppData\Local\Temp\ipykernel_24936\3431243700.py:64: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the futur